# Lab E — Make It Fast

**GPU Mastery · CUDA** · run on the **2× RTX 5090** server (or a Colab GPU)

Three levers turn a working kernel into a fast one — you'll **measure each yourself**:
1. **Coalescing** — watch bandwidth collapse as memory access gets scattered.
2. **Occupancy** — sweep the block size and find the sweet spot.
3. **Streams** — overlap copy with compute and measure the speedup.

Raw CUDA C++ via `nvcc` (same as Lab D). Compile with `-arch=native` (fallback: `-arch=sm_120`).

In [ ]:
!nvcc --version | tail -2
!nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader

## 1. Coalescing — the 1-vs-32 transaction, measured

Each thread reads one element with a configurable **stride**. Stride 1 = consecutive threads read consecutive addresses (**coalesced**, one transaction). Bigger strides scatter the reads across memory. Watch **GB/s fall** as the stride grows.

In [ ]:
%%writefile coalesce.cu
#include <cstdio>

__global__ void strided_copy(const float* in, float* out, int n, int stride) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) out[i] = in[((long)i * stride) % n];   // coalesced write, strided read
}

int main() {
    int n = 1 << 24;                                  // 16M elements
    size_t bytes = (size_t)n * sizeof(float);
    float *din, *dout;
    cudaMalloc(&din, bytes); cudaMalloc(&dout, bytes); cudaMemset(din, 1, bytes);

    int threads = 256, blocks = (n + threads - 1) / threads;
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);

    int strides[] = {1, 2, 4, 8, 16, 32};
    printf("%8s %12s %10s\n", "stride", "GB/s", "vs stride1");
    double base = 0;
    for (int k = 0; k < 6; k++) {
        int st = strides[k];
        strided_copy<<<blocks, threads>>>(din, dout, n, st); cudaDeviceSynchronize();  // warm
        cudaEventRecord(s);
        for (int r = 0; r < 20; r++) strided_copy<<<blocks, threads>>>(din, dout, n, st);
        cudaEventRecord(e); cudaEventSynchronize(e);
        float ms; cudaEventElapsedTime(&ms, s, e); ms /= 20;
        double gbps = (double)n * sizeof(float) * 2 / (ms / 1e3) / 1e9;   // read + write
        if (k == 0) base = gbps;
        printf("%8d %12.1f %9.2fx\n", st, gbps, gbps / base);
    }
    cudaFree(din); cudaFree(dout);
    return 0;
}

In [ ]:
!nvcc -arch=native -O2 coalesce.cu -o coalesce && ./coalesce

**Expect:** stride 1 hits near the card's peak bandwidth; by stride 32 it can be **many times slower** — same amount of useful data, but scattered reads need far more memory transactions. *This is the #1 memory lever.*

## 2. Occupancy — sweep the block size

The same light kernel, launched with different **threads-per-block**. Too few (32) and the SM can't hide latency; the very largest can hit resource limits. Find the fastest.

In [ ]:
%%writefile occupancy.cu
#include <cstdio>

__global__ void work(const float* in, float* out, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) { float v = in[i]; for (int k = 0; k < 32; k++) v = v * 1.001f + 0.5f; out[i] = v; }
}

int main() {
    int n = 1 << 24; size_t bytes = (size_t)n * sizeof(float);
    float *din, *dout;
    cudaMalloc(&din, bytes); cudaMalloc(&dout, bytes); cudaMemset(din, 1, bytes);
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);

    int bs[] = {32, 64, 128, 256, 512, 1024};
    printf("%10s %10s\n", "block", "ms");
    for (int i = 0; i < 6; i++) {
        int threads = bs[i], blocks = (n + threads - 1) / threads;
        work<<<blocks, threads>>>(din, dout, n); cudaDeviceSynchronize();   // warm
        cudaEventRecord(s);
        for (int r = 0; r < 30; r++) work<<<blocks, threads>>>(din, dout, n);
        cudaEventRecord(e); cudaEventSynchronize(e);
        float ms; cudaEventElapsedTime(&ms, s, e); ms /= 30;
        printf("%10d %10.3f\n", threads, ms);
    }
    cudaFree(din); cudaFree(dout);
    return 0;
}

In [ ]:
!nvcc -arch=native -O2 occupancy.cu -o occupancy && ./occupancy

**Expect:** a **U-shape** — very small blocks (32) starve the SM of warps to hide latency; the middle (128–256) is usually fastest. This is why *"tune the block size"* is the occupancy fix.

## 3. Streams — overlap copy with compute

Serial does copy-in → compute → copy-out for each chunk, one after another. With **streams + async copies + pinned memory**, the copies of one chunk hide behind the compute of another. Measure the wall-clock win.

In [ ]:
%%writefile streams.cu
#include <cstdio>

__global__ void busy(float* d, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) { float v = d[i]; for (int k = 0; k < 200; k++) v = v * 1.0001f + 0.1f; d[i] = v; }
}

int main() {
    int n = 1 << 22, chunks = 8;                          // 4M per chunk, 8 chunks
    size_t cb = (size_t)n * sizeof(float), tot = cb * chunks;
    float *h, *d;
    cudaMallocHost(&h, tot);                              // PINNED host memory (needed for async)
    cudaMalloc(&d, tot);
    for (size_t i = 0; i < (size_t)n * chunks; i++) h[i] = 1.0f;
    int threads = 256, blocks = (n + threads - 1) / threads;
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);

    // --- serial (default stream) ---
    cudaEventRecord(s);
    for (int c = 0; c < chunks; c++) {
        cudaMemcpy(d + (size_t)c * n, h + (size_t)c * n, cb, cudaMemcpyHostToDevice);
        busy<<<blocks, threads>>>(d + (size_t)c * n, n);
        cudaMemcpy(h + (size_t)c * n, d + (size_t)c * n, cb, cudaMemcpyDeviceToHost);
    }
    cudaDeviceSynchronize();
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms_serial; cudaEventElapsedTime(&ms_serial, s, e);

    // --- pipelined with 4 streams ---
    int nS = 4; cudaStream_t st[4]; for (int i = 0; i < nS; i++) cudaStreamCreate(&st[i]);
    cudaEventRecord(s);
    for (int c = 0; c < chunks; c++) {
        cudaStream_t sc = st[c % nS];
        cudaMemcpyAsync(d + (size_t)c * n, h + (size_t)c * n, cb, cudaMemcpyHostToDevice, sc);
        busy<<<blocks, threads, 0, sc>>>(d + (size_t)c * n, n);
        cudaMemcpyAsync(h + (size_t)c * n, d + (size_t)c * n, cb, cudaMemcpyDeviceToHost, sc);
    }
    cudaDeviceSynchronize();
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms_stream; cudaEventElapsedTime(&ms_stream, s, e);

    printf("serial   : %7.2f ms\n", ms_serial);
    printf("streams  : %7.2f ms\n", ms_stream);
    printf("speedup  : %7.2fx\n", ms_serial / ms_stream);
    for (int i = 0; i < nS; i++) cudaStreamDestroy(st[i]);
    cudaFreeHost(h); cudaFree(d);
    return 0;
}

In [ ]:
!nvcc -arch=native -O2 streams.cu -o streams && ./streams

**Expect:** the streamed version is **faster** — the copies overlap the compute instead of blocking it. The speedup won't be huge here (it's bounded by `max(compute, copy)`), but on copy-heavy pipelines it's large.

## Reflection (write your answers)

1. In Part 1, how many **× slower** was stride 32 vs stride 1? Explain in terms of memory transactions.
2. In Part 2, which **block size** was fastest? Why are 32 and 1024 not the best?
3. In Part 3, what **speedup** did streams give? Why isn't it larger? (hint: `max(compute, copy)`)
4. Match each result to a row in the **diagnosis → fix** table from the lecture.

### Cleanup

In [ ]:
!rm -f coalesce occupancy streams coalesce.cu occupancy.cu streams.cu
!echo cleaned